<a href="https://colab.research.google.com/github/teoalcdor/trabajo_iae/blob/main/minicpm_o_2_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniCPM-o-2.6:

Prueba de uso de MiniCPM-o-2.6, que podemos encontrar [aquí](https://github.com/OpenBMB/MiniCPM-o). En principio parece algo lento a la hora procesar cada página, pero es muy preciso a la hora de extraer texto de imágenes y tablas y merece consideración. Lo peor es que, de momento, requiere una versión de transformers antigua y muy concreta, lo que puede hacer que sea más difícil de integrar en el código que ya tenemos o que construyamos en un futuro.

## Entorno Virtual

Por si se desea acceder a Drive más tarde, lo montamos antes de crear y usar el entorno virtual:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Creamos un entorno virutual de Conda. Los paquetes que vienen pre-instalados en Colab, con sus correspondientes versiones, pueden dar problemas al instalar los requisitos, pero buscamos aprovechar las instancias de GPU y TPU de Colab.

In [ ]:
%env PYTHONPATH=
!pip install virtualenv
!virtualenv myenv
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda install -q -y --prefix /usr/local python=3.11.13 ujson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 36.6 MB/s eta 0:00:00
created virtual environment CPython3.11.13.final.0-64 in 331ms
  creator CPython3Posix(dest=/content/myenv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, setuptools=bundle, via=copy, app_data_dir=/root/.local/share/virtualenv)
    added seed packages: pip==25.1.1, setuptools==80.3.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator
--2025-07-17 06:49:40--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.32.241, 104.16.191.158, 2606:4700::6810:20f1, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.32.241|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 159476510 (152M) [application/octet-stream]
Saving to:

Activamos el entorno y pasamos a usarlo:

In [ ]:
import sys
import os
os.environ['CONDA_PREFIX'] = '/usr/local/envs/myenv'
sys.path.append('/usr/local/lib/python3.11.13/site-packages/')

Comprobamos que, efectivamente, no estamos usando el entorno proporcionado por Colab:

In [ ]:
!pip list

Package                  Version
------------------------ ---------
anaconda-anon-usage      0.7.1
annotated-types          0.6.0
archspec                 0.2.3
boltons                  25.0.0
brotlicffi               1.0.9.2
certifi                  2025.7.14
cffi                     1.17.1
charset-normalizer       3.3.2
conda                    25.5.1
conda-anaconda-telemetry 0.2.0
conda-anaconda-tos       0.2.0
conda-content-trust      0.2.0
conda-libmamba-solver    25.4.0
conda-package-handling   2.4.0
conda_package_streaming  0.12.0
cryptography             45.0.3
distro                   1.9.0
frozendict               2.4.2
idna                     3.7
jsonpatch                1.33
jsonpointer              2.1
libmambapy               2.0.5
markdown-it-py           2.2.0
mdurl                    0.1.0
menuinst                 2.3.0
packaging                24.2
pip                      25.1
platformdirs             4.3.7
pluggy                   1.5.0
pycosat                  0.6

## Descarga y Uso de MiniCPM-o-2.6

Instalamos los requerimientos de MiniCPM-o-2.6:

In [ ]:
!pip install transformers==4.44.2
!pip install --upgrade pip wheel setuptools packaging ninja
!pip install torch=='2.4.1+cu124' torchvision=='0.19.1+cu124' torchaudio=='2.4.1+cu124' \
    --index-url https://download.pytorch.org/whl/cu121
!pip install flash-attn --no-build-isolation

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torch-2.4.1%2Bcu121-cp311-cp311-linux_x86_64.whl (799.0 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchvision-0.19.1%2Bcu121-cp311-cp311-linux_x86_64.whl (7.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchaudio-2.4.1%2Bcu121-cp311-cp311-linux_x86_64.whl (3.4 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl (664.8 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 54.9 MB/s eta 0:00:00
  

Usamos el modelo:

In [ ]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained(
    'openbmb/MiniCPM-o-2_6',
    trust_remote_code=True,
    attn_implementation='sdpa', # sdpa or flash_attention_2
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=False,
    init_tts=False
)


model = model.eval().cuda()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-o-2_6', trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at openbmb/MiniCPM-o-2_6 were not used when initializing MiniCPMO: ['apm.conv1.bias', 'apm.conv1.weight', 'apm.conv2.bias', 'apm.conv2.weight', 'apm.embed_positions.weight', 'apm.layer_norm.bias', 'apm.layer_norm.weight', 'apm.layers.0.fc1.bias', 'apm.layers.0.fc1.weight', 'apm.layers.0.fc2.bias', 'apm.layers.0.fc2.weight', 'apm.layers.0.final_layer_norm.bias', 'apm.layers.0.final_layer_norm.weight', 'apm.layers.0.self_attn.k_proj.weight', 'apm.layers.0.self_attn.out_proj.bias', 'apm.layers.0.self_attn.out_proj.weight', 'apm.layers.0.self_attn.q_proj.bias', 'apm.layers.0.self_attn.q_proj.weight', 'apm.layers.0.self_attn.v_proj.bias', 'apm.layers.0.self_attn.v_proj.weight', 'apm.layers.0.self_attn_layer_norm.bias', 'apm.layers.0.self_attn_layer_norm.weight', 'apm.layers.1.fc1.bias', 'apm.layers.1.fc1.weight', 'apm.layers.1.fc2.bias', 'apm.layers.1.fc2.weight', 'apm.layers.1.final_layer_norm.bias', 'apm.layers.1.final_layer_norm.weight', 'apm.layers.1

La imagen muestra páginas de un libro con varios textos y fotografías en blanco y negro relacionadas con el toro. Aquí está el texto legible:

**Texto izquierdo:**
"La más arriesgada posibilidad del toro natural es con el capote, sean con la mano izquierda o con la derecha. Eduardo Solórzano en la plaza mexicana de El Torreón La Condesa"

**Texto derecho:**
"Faro con todo el pecho por delante del mexicano Antonio Velázquez

José María Mamzánares ha aportado a su chicle una toreadura mano baja impecable y mudana

La cordobina, creación de Jesús Córdoba, en versión de Rafael Yagüe"


In [ ]:
# 1️⃣ Abre la imagen (texto impreso, documento escaneado, foto, etc.)
img = Image.open("img_3.jpg").convert("RGB")

# 2️⃣ Prepara el prompt de texto
# La pregunta orienta al modelo a extraer el texto
prompt = "Por favor extrae todo el texto legible de la siguiente imagen:"

# 3️⃣ Usa el formato chat del modelo para enviar la imagen y la solicitud
msgs = [
    {"role": "user", "content": [img, prompt]}
]

# 4️⃣ Ejecuta la inferencia
respuesta = model.chat(
    msgs=msgs,
    tokenizer=tokenizer
)

# 5️⃣ Imprime la respuesta del modelo
print(respuesta)

**Título:**
el RD1178/2023 de 27 de diciembre. En resumen, los valores de ayudas establecidos son los siguientes:

**Subtítulo:**
Gran empresa Median empresa Pequeña empresa

**Tabla:**
| Programa | Instalación autoconsumo | Instalación almacenamiento |
|----------|-------------------------|--------------------------|
| Programa 1 y 2 | 15% - 30% | 25% - 40% | 35% - 50% |
| Programa 1, 2 y 3 | 30% | 40% | 50% |

**Texto:**
El sector servicios y otros sectores productivos con incentivos para instalaciones de autoconsumo con energía solar fotovoltaica y eólica que oscilan entre el 15% y el 50% en función de la tecnología, del tamaño de la empresa y de la potencia de la instalación.

Si estos proyectos llevan asociado almacenamiento, las ayuda para este equipo se situaran entre el 45% y el 65% de la inversión que se realice, según el tamaño de la empresa. Estos incentivos también se pueden solicitar para instalaciones de autoconsumo ya existentes.

**Subtítulo:**
Programa 4

**Texto:**
Se